# Restaurant Branch Performance — Exploratory Data Analysis

Download the Restaurant Branch Performance CSV from the LMS **Study Material** tab. Run the one code cell below and upload the dataset when prompted.

In [ ]:
# Restaurant Branch Performance EDA — single Google Colab cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the Restaurant Branch Performance CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip()
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

def find_column(candidates):
    normalized = {col.lower().replace('_', ' ').replace('-', ' ').strip(): col for col in df.columns}
    for candidate in candidates:
        if candidate in normalized:
            return normalized[candidate]
    for name, original in normalized.items():
        if any(candidate in name for candidate in candidates):
            return original
    return None

branch_col = find_column(['branch', 'branch name', 'restaurant branch', 'store'])
region_col = find_column(['region', 'area', 'location'])
store_type_col = find_column(['store type', 'restaurant type', 'type'])
revenue_col = find_column(['revenue', 'sales', 'total sales'])
profit_col = find_column(['profit', 'net profit'])
customers_col = find_column(['customers', 'customer count', 'customer'])
marketing_col = find_column(['marketing spend', 'marketing', 'advertising spend'])
staff_col = find_column(['staff count', 'staff', 'employee count'])
delivery_col = find_column(['delivery time', 'delivery'])
rating_col = find_column(['customer rating', 'rating', 'ratings'])

# Convert expected measures to numeric, keeping non-convertible values as missing.
measure_columns = [col for col in [revenue_col, profit_col, customers_col, marketing_col, staff_col, delivery_col, rating_col] if col]
for column in measure_columns:
    df[column] = pd.to_numeric(df[column].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')

print('=' * 90)
print(f'RESTAURANT BRANCH PERFORMANCE EDA: {csv_files[0]}')
print('=' * 90)

print('\n1. DATASET OVERVIEW')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Columns:', df.columns.tolist())
display(df.head())
print('\nData types and non-null counts:')
df.info()
print('\nMissing values:')
display(pd.DataFrame({'Missing Values': df.isnull().sum(), 'Missing (%)': (df.isnull().mean() * 100).round(2)}))

print('\n2. SUMMARY AND DESCRIPTIVE STATISTICS')
display(df.describe(include='all').T)
if measure_columns:
    summary = df[measure_columns].agg(['count', 'mean', 'median', 'min', 'max', 'std']).T.round(2)
    display(summary)

print('\n3. DISTRIBUTION ANALYSIS')
plot_columns = measure_columns[:7]
if plot_columns:
    rows = int(np.ceil(len(plot_columns) / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(14, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for axis, column in zip(axes, plot_columns):
        sns.histplot(df[column].dropna(), kde=True, ax=axis, color='#247BA0')
        axis.set_title(f'Distribution of {column}')
    for axis in axes[len(plot_columns):]:
        axis.set_visible(False)
    plt.tight_layout()
    plt.show()

def performance_summary(group_column, title):
    if not group_column:
        print(f'\n{title}: relevant grouping column not found.')
        return None
    available_measures = [column for column in [revenue_col, profit_col, customers_col, marketing_col, staff_col, delivery_col, rating_col] if column]
    if not available_measures:
        print(f'\n{title}: no numeric performance measure was identified.')
        return None
    summary = df.groupby(group_column)[available_measures].mean().round(2).sort_values(revenue_col if revenue_col else available_measures[0], ascending=False)
    summary['Records'] = df.groupby(group_column).size()
    print(f'\n{title}')
    display(summary)
    return summary

print('\n4. PERFORMANCE COMPARISON')
branch_summary = performance_summary(branch_col, 'Average performance by branch')
region_summary = performance_summary(region_col, 'Average performance by region')
store_type_summary = performance_summary(store_type_col, 'Average performance by store type')

if revenue_col and branch_col:
    plt.figure(figsize=(12, 6))
    branch_order = df.groupby(branch_col)[revenue_col].mean().sort_values(ascending=False).index
    sns.barplot(data=df, x=branch_col, y=revenue_col, order=branch_order, estimator='mean', errorbar=None, color='#70C1B3')
    plt.title('Average Revenue by Branch')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

print('\n5. CORRELATION ANALYSIS')
numeric_data = df.select_dtypes(include=np.number)
if numeric_data.shape[1] >= 2:
    correlation = numeric_data.corr(numeric_only=True)
    display(correlation.round(2))
    plt.figure(figsize=(max(8, numeric_data.shape[1]), max(6, numeric_data.shape[1] * 0.7)))
    sns.heatmap(correlation, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f', square=True)
    plt.title('Correlation Matrix of Numerical Variables')
    plt.tight_layout()
    plt.show()
else:
    correlation = None
    print('At least two numeric columns are needed for correlation analysis.')

print('\n6. KEY FINDINGS')
observations = [
    f'1. The dataset contains {len(df):,} records and {len(df.columns)} variables.',
    f'2. There are {df.isnull().sum().sum():,} missing values across the dataset.',
]
if revenue_col:
    observations.append(f'3. Average revenue is {df[revenue_col].mean():,.2f}, ranging from {df[revenue_col].min():,.2f} to {df[revenue_col].max():,.2f}.')
if profit_col:
    observations.append(f'{len(observations) + 1}. Average profit is {df[profit_col].mean():,.2f}; the highest recorded profit is {df[profit_col].max():,.2f}.')
if branch_summary is not None and revenue_col:
    observations.append(f'{len(observations) + 1}. The top branch by average revenue is {branch_summary.index[0]} ({branch_summary.iloc[0][revenue_col]:,.2f}).')
if region_summary is not None and revenue_col:
    observations.append(f'{len(observations) + 1}. The top region by average revenue is {region_summary.index[0]} ({region_summary.iloc[0][revenue_col]:,.2f}).')
if store_type_summary is not None and revenue_col:
    observations.append(f'{len(observations) + 1}. The highest-performing store type by average revenue is {store_type_summary.index[0]} ({store_type_summary.iloc[0][revenue_col]:,.2f}).')
if correlation is not None:
    pairs = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool)).stack()
    if not pairs.empty:
        (first_variable, second_variable), correlation_value = pairs.abs().idxmax(), pairs.loc[pairs.abs().idxmax()]
        direction = 'positive' if correlation_value > 0 else 'negative'
        observations.append(f'{len(observations) + 1}. The strongest linear relationship is between {first_variable} and {second_variable} ({direction} correlation: {correlation_value:.2f}).')
for observation in observations[:8]:
    print(observation)